##  Bronze to Silver Job

- 1. Verify the column names and standardize to snake_case
- 2. Treat the city administrative descriptors columns 
- 3. Treat the survey indicators from "Informática e Comunicação" worksheet
- 4. Treat the survey indicators from "Governanca" worksheet
- 5. Integrity Check 
    - primary key uniqueness and row count match (5,570 == 5,570)
- 6. Merge data tables
- 7. Write to silver

In [0]:
%run ../environment/env_variables

## Envirnoment Variables

- This notebook is made to be imported by the jobs of the Munic Pipeline
- The objective is set common variables used between the pipeline to easy the management of them
and centralize the definition in one place, avoiding typo errors along the pipeline

In [0]:
%run ../environment/env_functions

# env_functions

- This notebook is to sabe methods used along all the Munic Pipeline

In [0]:
import pyspark.sql.functions as F

In [0]:
# Paths of where the final tables are going to be stored
bronze_tbl_dict = f'{catalog_name}.{bronze_schema}.munic_raw_dict'
bronze_tbl_info = f'{catalog_name}.{bronze_schema}.munic_raw_info_data'
bronze_tbl_gov = f'{catalog_name}.{bronze_schema}.munic_raw_gov_data'

silver_tbl_dict = f'{catalog_name}.{silver_schema}.munic_clean_dict'
silver_tbl_data = f'{catalog_name}.{silver_schema}.munic_clean_data'

## Step 1

In [0]:
# Load bronze tables, filtered by the survey edition year
ano_edicao = int(reference_year) if isinstance(reference_year, (int, float, str)) else 2024

df_raw_dict = spark.table(bronze_tbl_dict).filter(F.col("ano_edicao") == ano_edicao)
df_raw_info = spark.table(bronze_tbl_info).filter(F.col("ano_edicao") == ano_edicao)
df_raw_gov = spark.table(bronze_tbl_gov).filter(F.col("ano_edicao") == ano_edicao)

In [0]:
# Verify that all column names are standardized to snake_case
import re

# Define a regex expression to match snake_case
regex_expr = r'^[a-z0-9_]+$'

for name, df in [("munic_raw_dict", df_raw_dict), ("munic_raw_info", df_raw_info), ("munic_raw_gov", df_raw_gov)]:
    non_snake = [col for col in df.columns if not re.match(regex_expr, col)]

    print(f"{name}: {len(df.columns)} columns, non-snake_case: {non_snake or 'none'}")

munic_raw_dict: 7 columns, non-snake_case: none
munic_raw_info: 210 columns, non-snake_case: none
munic_raw_gov: 174 columns, non-snake_case: none


## Step 2 

In [0]:
df_city_metainfo = df_raw_info.select("cod_munic", "desc_mun", "uf", "populacao", "faixa_populacao", "regiao")

In [0]:
df_city_metainfo.printSchema()

root
 |-- cod_munic: long (nullable = true)
 |-- desc_mun: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- populacao: long (nullable = true)
 |-- faixa_populacao: string (nullable = true)
 |-- regiao: string (nullable = true)



In [0]:
# 1. Transform the cod_munic into string 
# 2. Drop the column 'cod_munic'
# 3. Rename columns for clarity
def treat_city_metainfo(df):
    return (
        df
        .withColumn("cod_ibge", F.col("cod_munic").cast("string"))
        .drop("cod_munic")
        .withColumnsRenamed({
            "desc_mun": "nome",
            "uf": "uf_sigla",
            "populacao": "populacao_total",
        })
    )


In [0]:
# Apply the treatment to the city metainfo
df_city_metainfo = treat_city_metainfo(df_city_metainfo)
display(df_city_metainfo.limit(5))

nome,uf_sigla,populacao_total,faixa_populacao,regiao,cod_ibge
Alta Floresta DOeste,RO,22853,4 - 20001 até 50000,1 - Norte,1100015
Ariquemes,RO,108573,6 - 100001 até 500000,1 - Norte,1100023
Cabixi,RO,5690,2 - 5001 até 10000,1 - Norte,1100031
Cacoal,RO,97637,5 - 50001 até 100000,1 - Norte,1100049
Cerejeiras,RO,16975,3 - 10001 até 20000,1 - Norte,1100056


## Step 3 - Treat the survey indicators from "Informática e Comunicação" worksheet

In [0]:
# Select only cod_munic and the MTIC indicator columns from the info table
# Drop the city descriptor and metadata columns
mtic_cols = [c for c in df_raw_info.columns if c.startswith("mtic")]
df_info_indicators = df_raw_info.select("cod_munic", *mtic_cols)

print(f"Info indicators: {len(mtic_cols)} MTIC columns")
display(df_info_indicators.limit(5))

Info indicators: 198 MTIC columns


cod_munic,mtic011,mtic012,mtic013,mtic014,mtic015,mtic016,mtic017,mtic018,mtic04,mtic06,mtic071,mtic072,mtic073,mtic074,mtic075,mtic076,mtic077,mtic078,mtic08,mtic091,mtic092,mtic093,mtic10,mtic11,mtic12a1,mtic12a2,mtic12a3,mtic12a4,mtic12a5,mtic12a6,mtic12a7,mtic12a8,mtic12b1,mtic12b2,mtic12b3,mtic12b4,mtic12b5,mtic12b6,mtic12b7,mtic12b8,mtic12b9,mtic12b10,mtic12b11,mtic12b12,mtic12b13,mtic12b14,mtic131,mtic132,mtic133,mtic134,mtic135,mtic136,mtic137,mtic138,mtic139,mtic141,mtic142,mtic143,mtic144,mtic145,mtic146,mtic15,mtic161,mtic162,mtic163,mtic164,mtic165,mtic166,mtic167,mtic171,mtic172,mtic173,mtic174,mtic175,mtic176,mtic181,mtic182,mtic183,mtic184,mtic185,mtic186,mtic18a1,mtic18a2,mtic18a3,mtic18a4,mtic18a5,mtic18a6,mtic19,mtic201,mtic202,mtic203,mtic204,mtic205,mtic206,mtic211,mtic212,mtic213,mtic214,mtic215,mtic216,mtic217,mtic22,mtic23,mtic241,mtic242,mtic2421,mtic243,mtic2431,mtic244,mtic251,mtic2511,mtic2512,mtic2513,mtic2514,mtic252,mtic2521,mtic2522,mtic2523,mtic2524,mtic253,mtic2531,mtic2532,mtic2533,mtic2534,mtic254,mtic2541,mtic2542,mtic2543,mtic2544,mtic255,mtic261,mtic262,mtic263,mtic264,mtic265,mtic266,mtic271,mtic272,mtic273,mtic281,mtic282,mtic283,mtic284,mtic285,mtic286,mtic287,mtic288,mtic289,mtic29,mtic30,mtic31,mtic321,mtic322,mtic323,mtic324,mtic325,mtic326,mtic327,mtic328,mtic33,mtic331,mtic341,mtic342,mtic343,mtic344,mtic345,mtic346,mtic347,mtic348,mtic349,mtic35,mtic3511,mtic3512,mtic3513,mtic3514,mtic3515,mtic3516,mtic3517,mtic3518,mtic3519,mtic35110,mtic35111,mtic361,mtic362,mtic363,mtic364,mtic365,mtic366,mtic367,mtic368,mtic37,mtic381,mtic382,mtic383,mtic384,mtic385,mtic386,mtic387
1100015,Não,Não,Sim,Sim,Sim,Não,Não,Não,Via cabo ou fibra ótica,Não,-,-,-,-,-,-,-,-,Não,Não,Não,Sim,Ativa,WWW.ALTAFLORESTA.RO.GOV.BR,Sim,Não,Sim,Sim,Sim,Sim,Sim,Não,Sim,Não,Sim,Não,Não,Sim,Não,Sim,Não,Não,Sim,Não,Não,Não,Sim,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Diariamente,Não,Não,Não,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,Gratuito,Não,Sim,Parte,Não,-,Não,Não,-,-,-,-,Não,-,-,-,-,Não,-,-,-,-,Não,-,-,-,-,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Modelo próprio de acessibilidade digital,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,Sim,Não,Não,Não,Não,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Sim,Não,Não,Não
1100023,Não,Não,Sim,Sim,Sim,Não,Não,Não,Via cabo ou fibra ótica,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Não,Sim,Não,Não,Sim,Ativa,WWW.ARIQUEMES.RO.GOV.BR,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Não,Não,Sim,Não,Não,Não,Sim,Sim,Não,Sim,Não,Não,Diariamente,Não,Não,Não,Não,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Não,Sim,Sim,Sim,Não,Sim,Não,Sim,Não,Sim,Não,Não,Não,Sim,Não,Sim,Não,Não,Não,Não,Sim,Não,Sim,Sim,Sim,Sim,Não,Sim,Gratuito,Não,Não,-,Não,-,Sim,Sim,Não,Não,Não,Sim,Não,-,-,-,-,Sim,Sim,Não,Não,Não,Não,-,-,-,-,Não,Sim,Não,Sim,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Modelo global de acessibilidade digital (WCAG/WAI),Sim,DEPARTAMENTO DE COMUNICAÇÃO,Não,Não,Não,Sim,Não,Não,Não,Não,Não,-,Não,Não,Sim,Não,Sim,Não,Sim,Não,Não,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,Não,Não,Não,Não,Sim,Não,Não
1100031,Não,Não,Sim,Não,Sim,Não,Não,Não,Via cabo ou fibra ótica,Não,-,-,-,-,-,-,-,-,Sim,Não,Não,Sim,Ativa,WWW.CABIXI.RO.GOV.BR,Sim,Sim,Sim,Não,Sim,Sim,Sim,Não,Sim,Sim,Sim,Não,Não,Sim,Não,Sim,Não,Não,Sim,Sim,Sim,Não,Não,Não,Não,Não,Não,Não,Não,Não,Sim,-,-,-,-,-,-,-,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Não,Não,Sim,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,-,-,-,-,-,-,Não,-,-,-,-,Não,-,-,-,-,Não,-,-,-,-,Não,-,-,-,-,Sim,Não,Não,Não,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Não há qualquer modelo de acessibilidade digital,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,-,-,-,-,-,Sim
1100049,Não,Não,Sim,Sim,Sim,Não,Não,Não,Vi

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Reusable function: treat IBGE MUNIC survey indicator columns.
# Works for any worksheet (Informática, Governanca, etc.) given its column prefix.
# ─────────────────────────────────────────────────────────────────────────────

def treat_survey_indicators(df, column_prefix, dict_table_name):
    """
    Classify indicator columns as binary or categorical, then apply treatment.

    Binary      → Sim=True, Não=False, non-response=NULL
    Categorical → trim whitespace, non-response=NULL, keep as string

    Parameters
    ----------
    df               : Spark DataFrame with cod_munic + indicator columns
    column_prefix    : lowercase prefix ("mtic" or "mgov")
    dict_table_name  : fully qualified dictionary table name

    Returns a dict: df, binary_cols, categorical_cols, meanings
    """
    # Standard IBGE non-response codes to convert to NULL
    NON_RESPONSE = ["Recusa", "Não informou", "Não sabe informar", "Não sabe", "-"]

    # Load dictionary meanings for this prefix (codes stored uppercase in dict)
    df_dict = spark.table(dict_table_name)
    meanings = {
        row["cod_variable"].lower(): row["meaning"]
        for row in df_dict.filter(
            F.col("cod_variable").rlike(f"^{column_prefix.upper()}")
        ).collect()
    }
    print(f"Loaded {len(meanings)} {column_prefix.upper()} meanings from dictionary")

    # Cache column list once (avoids repeated Analyze RPCs on Spark Connect)
    indicator_cols = [c for c in df.columns if c.startswith(column_prefix)]

    # Single-pass collect_set to get all distinct values per column (1 Spark job)
    distinct_sets = df.agg(
        *[F.collect_set(c).alias(c) for c in indicator_cols]
    ).collect()[0].asDict()

    # Classify: subset of {"Sim", "Não"} → binary, otherwise categorical
    binary_cols, categorical_cols = [], []
    for col_name in indicator_cols:
        meaningful = set(str(v) for v in distinct_sets[col_name]) - set(NON_RESPONSE) - {"None", "nan"}
        if meaningful <= {"Sim", "Não"}:
            binary_cols.append(col_name)
        else:
            categorical_cols.append(col_name)

    print(f"Binary: {len(binary_cols)} | Categorical: {len(categorical_cols)}")
    print(f"Categorical: {categorical_cols}")

    # Build all transformations as a single select() (flat plan, no nested withColumn)
    select_exprs = [F.col("cod_munic")]
    for col_name in indicator_cols:
        if col_name in binary_cols:
            # Sim/Não → boolean, everything else → NULL
            select_exprs.append(
                F.when(F.col(col_name) == "Sim", True)
                .when(F.col(col_name) == "Não", False)
                .otherwise(None)
                .alias(col_name)
            )
        else:
            # Trim whitespace; non-response codes → NULL; otherwise keep value
            select_exprs.append(
                F.when(F.trim(F.col(col_name)).isin(NON_RESPONSE), None)
                .otherwise(F.trim(F.col(col_name)))
                .alias(col_name)
            )

    treated_df = df.select(*select_exprs)

    return {
        "df": treated_df,
        "binary_cols": binary_cols,
        "categorical_cols": categorical_cols,
        "meanings": meanings,
    }

#### Treating applying in indicators

In [0]:
# Apply the reusable treatment function to MTIC (Informática e Comunicação) indicators
# column_prefix="mtic" matches columns like mtic011, mtic04, mtic12a1, etc.
# dict_table_name=bronze_tbl_dict is the munic_raw_dict table defined in cell 4
result_info = treat_survey_indicators(
    df_info_indicators,
    column_prefix="mtic",
    dict_table_name=bronze_tbl_dict,
    ano_edicao=ano_edicao
)

# Unpack the results dict into individual variables
df_info_indicators = result_info["df"]                   # treated DataFrame (binary→boolean, etc.)
mtic_meanings = result_info["meanings"]                 # dict: {"mtic011": "Correio", ...}
binary_cols_info = result_info["binary_cols"]           # list of 188 binary column names
categorical_cols_info = result_info["categorical_cols"]  # list of 10 categorical column names

print(f"\nTreatment applied to {len(binary_cols_info) + len(categorical_cols_info)} MTIC columns")

# Verify the treatment: schema changes and null counts
print("=== Schema after treatment (first 15 MTIC columns) ===")
df_info_indicators.printSchema()

# Show null counts for a sample of binary and categorical columns
# This confirms that non-response codes (Recusa, Não informou, -, etc.) were converted to NULL
print("\n=== Null counts (non-response codes converted to NULL) ===")
sample_check = ["mtic011", "mtic04", "mtic10", "mtic11", "mtic15", "mtic29", "mtic31"]

# Count the rows ONCE, outside the loop. Calling .count() inside the loop would
# trigger one full Spark job per iteration just to recompute the same number.
total = df_info_indicators.count()

for col_name in sample_check:
    null_count = df_info_indicators.filter(F.col(col_name).isNull()).count()
    meaning = mtic_meanings.get(col_name, '?')[:50]
    print(f"  {col_name} ({meaning}): {null_count}/{total} nulls")

Loaded 197 MTIC meanings from dictionary
Binary (Sim/Não → boolean): 188
Categorical/text (→ cleaned string): 10
Categorical: ['mtic04', 'mtic10', 'mtic11', 'mtic15', 'mtic23', 'mtic2421', 'mtic2431', 'mtic29', 'mtic31', 'mtic331']

Treatment applied to 198 MTIC columns
=== Schema after treatment (first 15 MTIC columns) ===
root
 |-- cod_munic: long (nullable = true)
 |-- mtic011: boolean (nullable = true)
 |-- mtic012: boolean (nullable = true)
 |-- mtic013: boolean (nullable = true)
 |-- mtic014: boolean (nullable = true)
 |-- mtic015: boolean (nullable = true)
 |-- mtic016: boolean (nullable = true)
 |-- mtic017: boolean (nullable = true)
 |-- mtic018: boolean (nullable = true)
 |-- mtic04: string (nullable = true)
 |-- mtic06: boolean (nullable = true)
 |-- mtic071: boolean (nullable = true)
 |-- mtic072: boolean (nullable = true)
 |-- mtic073: boolean (nullable = true)
 |-- mtic074: boolean (nullable = true)
 |-- mtic075: boolean (nullable = true)
 |-- mtic076: boolean (nullable =

## Step 4 - Treat the survey indicators from "Governanca" worksheet

In [0]:
# Select only cod_munic and the MGOV indicator columns from the gov table
# Drop the city descriptor and metadata columns
mgov_cols = [c for c in df_raw_gov.columns if c.startswith("mgov")]
df_gov_indicators = df_raw_gov.select("cod_munic", *mgov_cols)

print(f"Gov indicators: {len(mgov_cols)} MGOV columns")
display(df_gov_indicators.limit(5))

Gov indicators: 162 MGOV columns


cod_munic,mgov01,mgov011a,mgov011b,mgov0121,mgov0122,mgov021,mgov022,mgov023,mgov024,mgov025,mgov026,mgov027,mgov028,mgov031,mgov032,mgov033,mgov034,mgov035,mgov036,mgov04,mgov05,mgov06,mgov061,mgov0621,mgov0622,mgov0623,mgov0711,mgov0712,mgov0713,mgov0715,mgov0721,mgov0722,mgov0723,mgov0725,mgov0731,mgov0732,mgov0733,mgov0735,mgov0741,mgov0742,mgov0743,mgov0745,mgov0751,mgov0752,mgov0753,mgov0755,mgov0761,mgov0762,mgov0763,mgov0765,mgov0771,mgov0772,mgov0773,mgov0775,mgov081,mgov082,mgov083,mgov084,mgov085,mgov086,mgov087,mgov088,mgov089,mgov0810,mgov0811,mgov09,mgov091,mgov092a,mgov092b,mgov10,mgov111,mgov112,mgov113,mgov114,mgov115,mgov116,mgov13,mgov141,mgov142,mgov15,mgov16,mgov17,mgov171,mgov172,mgov181,mgov182,mgov183,mgov184,mgov185,mgov186,mgov187,mgov188,mgov189,mgov1810,mgov1811,mgov1812,mgov1813,mgov1814,mgov1815,mgov1816,mgov1817,mgov1818,mgov191,mgov192,mgov193,mgov194,mgov195,mgov196,mgov201,mgov202,mgov203,mgov204,mgov211,mgov212,mgov213,mgov221,mgov222,mgov223,mgov224,mgov23,mgov241,mgov242,mgov243,mgov244,mgov245,mgov246,mgov247,mgov248,mgov251,mgov252,mgov253,mgov254,mgov255,mgov26,mgov261a,mgov261b,mgov261c,mgov262,mgov271,mgov272,mgov273,mgov274,mgov28,mgov29,mgov301,mgov302,mgov303,mgov31,mgov3111,mgov3112,mgov3113,mgov3114,mgov3115,mgov3116,mgov3117,mgov3118,mgov3119,mgov32,mgov33,mgov34,mgov35,mgov36
1100015,Não,-,-,-,-,-,-,-,-,-,-,-,-,Não,Sim,Não,Não,Sim,Não,Sim,Não publica,Sim,Não,Não,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Não,"Sim, instituída por instrumento legal",Lei,6,2017,Setor subordinado diretamente à chefia do Executivo,-,-,-,-,-,-,Sim,1,-,Servidor com cargo efetivo municipal,Ciências Contábeis,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Sim,Sim,Não,Não,Sim,Não,Sim,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Sim,Não,Não,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Não,-,Não,Não,Não
1100023,Sim,13412,2017,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Sim,Não,Não,Não,Sim,Não publica,Sim,Sim,Não,Sim,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Não,"Sim, instituída por instrumento legal",Lei,2248,2019,Secretaria municipal exclusiva,-,-,-,-,-,-,Sim,11,-,Servidor com cargo efetivo municipal,Ciências Contábeis,Sim,Não,-,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Não,Sim,Sim,Não,Sim,Sim,Sim,Não,Sim,Sim,Não,Não,Não,Não,Sim,Sim,Sim,Sim,Não,Sim,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,Na Controladoria ou Corregedoria municipal,Sim,Sim,Sim
1100031,Sim,148,2016,Não,Sim,Sim,Sim,Não,Sim,Não,Sim,Sim,Sim,Sim,Sim,Sim,Não,Não,Não,Não,Não publica,Sim,Sim,Não,Sim,Não,Não,Sim,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Não,"Sim, instituída por instrumento legal",Lei,437,2003,Setor subordinado diretamente à chefia do Executivo,-,-,-,-,-,-,Sim,1,-,"Servidor com cargo efetivo municipal, concursado para a área",Ciências Contábeis,Não,-,-,Sim,Sim,Não,Não,Não,Sim,Não,Não,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Sim,Não,Não,Sim,Sim,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Sim,Não,Sim,Não,Sim,Não,Sim,Não,Sim,Não,Não,Não,Não,Sim,Não,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,Sim,No gabinete do(a) Prefeito(a),Não,Sim,Sim
1100049,Sim,6544,2017,Não,Sim,Sim,Sim,Sim,Sim,Não,Sim,Sim,Sim,Sim,Não,Não,Não,Não,Não,Sim,Perguntas e respostas,Sim,Sim,Não,Sim,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Não,Sim,Não,Não,Sim,Sim,Não,Não,Sim,Não,Sim,Sim,Sim,Sim,Sim,Sim,Sim,Não,Não,"Sim, instituída por instrumento legal",Lei,3620,2016,Setor subordinado diretamente à chefia do Executivo,-,-,-,-,-,-,S

In [0]:
# Apply the SAME reusable treatment function to MGOV (Governanca) indicators
# column_prefix="mgov" matches columns like mgov01, mgov011a, mgov36, etc.
# We reuse the same function — no code duplication between MTIC and MGOV treatment
result_gov = treat_survey_indicators(
    df_gov_indicators,
    column_prefix="mgov",
    dict_table_name=bronze_tbl_dict,
    ano_edicao=ano_edicao
)

# Unpack the results
df_gov_indicators = result_gov["df"]                    # treated DataFrame
mgov_meanings = result_gov["meanings"]                 # dict: {"mgov01": "...", ...}
binary_cols_gov = result_gov["binary_cols"]             # list of binary column names
categorical_cols_gov = result_gov["categorical_cols"]  # list of categorical column names

print(f"\nTreatment applied to {len(binary_cols_gov) + len(categorical_cols_gov)} MGOV columns")
display(df_gov_indicators.limit(5))

Loaded 161 MGOV meanings from dictionary
Binary (Sim/Não → boolean): 144
Categorical/text (→ cleaned string): 18
Categorical: ['mgov011a', 'mgov011b', 'mgov05', 'mgov09', 'mgov091', 'mgov092a', 'mgov092b', 'mgov10', 'mgov141', 'mgov15', 'mgov16', 'mgov261a', 'mgov261b', 'mgov261c', 'mgov262', 'mgov28', 'mgov29', 'mgov33']

Treatment applied to 162 MGOV columns


cod_munic,mgov01,mgov011a,mgov011b,mgov0121,mgov0122,mgov021,mgov022,mgov023,mgov024,mgov025,mgov026,mgov027,mgov028,mgov031,mgov032,mgov033,mgov034,mgov035,mgov036,mgov04,mgov05,mgov06,mgov061,mgov0621,mgov0622,mgov0623,mgov0711,mgov0712,mgov0713,mgov0715,mgov0721,mgov0722,mgov0723,mgov0725,mgov0731,mgov0732,mgov0733,mgov0735,mgov0741,mgov0742,mgov0743,mgov0745,mgov0751,mgov0752,mgov0753,mgov0755,mgov0761,mgov0762,mgov0763,mgov0765,mgov0771,mgov0772,mgov0773,mgov0775,mgov081,mgov082,mgov083,mgov084,mgov085,mgov086,mgov087,mgov088,mgov089,mgov0810,mgov0811,mgov09,mgov091,mgov092a,mgov092b,mgov10,mgov111,mgov112,mgov113,mgov114,mgov115,mgov116,mgov13,mgov141,mgov142,mgov15,mgov16,mgov17,mgov171,mgov172,mgov181,mgov182,mgov183,mgov184,mgov185,mgov186,mgov187,mgov188,mgov189,mgov1810,mgov1811,mgov1812,mgov1813,mgov1814,mgov1815,mgov1816,mgov1817,mgov1818,mgov191,mgov192,mgov193,mgov194,mgov195,mgov196,mgov201,mgov202,mgov203,mgov204,mgov211,mgov212,mgov213,mgov221,mgov222,mgov223,mgov224,mgov23,mgov241,mgov242,mgov243,mgov244,mgov245,mgov246,mgov247,mgov248,mgov251,mgov252,mgov253,mgov254,mgov255,mgov26,mgov261a,mgov261b,mgov261c,mgov262,mgov271,mgov272,mgov273,mgov274,mgov28,mgov29,mgov301,mgov302,mgov303,mgov31,mgov3111,mgov3112,mgov3113,mgov3114,mgov3115,mgov3116,mgov3117,mgov3118,mgov3119,mgov32,mgov33,mgov34,mgov35,mgov36
1100015,false,null,null,null,null,null,null,null,null,null,null,null,null,false,true,false,false,true,false,true,Não publica,true,false,false,true,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,false,true,false,false,true,true,true,true,true,true,true,true,true,false,false,"Sim, instituída por instrumento legal",Lei,6,2017,Setor subordinado diretamente à chefia do Executivo,null,null,null,null,null,null,true,1,null,Servidor com cargo efetivo municipal,Ciências Contábeis,true,true,false,true,true,false,false,false,true,true,false,false,true,false,true,true,false,false,true,false,false,false,true,false,false,false,false,false,false,true,false,false,false,true,false,false,false,true,true,false,false,false,false,false,false,false,true,false,false,false,false,true,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,null,false,false,false
1100023,true,13412,2017,false,true,true,true,true,true,true,true,false,true,true,true,true,false,false,false,true,Não publica,true,true,false,true,false,true,true,false,false,true,true,false,false,true,true,false,false,true,true,false,false,true,true,false,false,true,true,false,false,true,true,false,false,true,true,true,true,true,true,true,true,true,false,false,"Sim, instituída por instrumento legal",Lei,2248,2019,Secretaria municipal exclusiva,null,null,null,null,null,null,true,11,null,Servidor com cargo efetivo municipal,Ciências Contábeis,true,false,null,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,true,false,true,true,false,true,true,false,true,true,true,false,true,true,false,false,false,false,true,true,true,true,false,true,true,false,false,true,false,false,false,true,false,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,true,Na Controladoria ou Corregedoria municipal,true,true,true
1100031,true,148,2016,false,true,true,true,false,true,false,true,true,true,true,true,true,false,false,false,false,Não publica,true,true,false,true,false,false,true,false,false,true,false,false,false,true,false,false,false,false,true,false,false,false,true,false,false,true,true,false,false,true,true,false,false,false,true,true,true,true,true,true,true,true,false,false,"Sim, instituída por instrumento legal",Lei,437,2003,Setor subordinado diretamente à chefia do Executivo,null,null,null,null,null,null,true,1,null,"Servidor com cargo efetivo municipal, concursado para a área",Ciências Contábeis,false,null,null,true,true,fals

## Step 5 - Integrity Check
- primary key uniqueness and row count match (5,570 == 5,570)

In [0]:
# Check primary key uniqueness in both source tables
info_count = df_raw_info.count()
gov_count = df_raw_gov.count()

# .distinct() is REQUIRED here: .select() alone does not deduplicate, so without it
# these would just recount the rows and the uniqueness assertions below would always
# pass, no matter how many duplicate keys the table actually holds.
info_distinct = df_raw_info.select("cod_munic").distinct().count()
gov_distinct = df_raw_gov.select("cod_munic").distinct().count()

print(f"Info table: {info_count} rows, {info_distinct} distinct cod_munic")
print(f"Gov table:  {gov_count} rows, {gov_distinct} distinct cod_munic")

assert info_count == info_distinct, f"Duplicate cod_munic found in info table"
assert gov_count == gov_distinct, f"Duplicate cod_munic found in gov table"
assert info_count == gov_count, f"Row count mismatch: {info_count} != {gov_count}"
assert info_count == 5570, f"Expected 5.570 rows, got {info_count}"

print(f"\nAll integrity checks passed: {info_count} == {gov_count} == 5.570 rows")

Info table: 5570 rows, 5570 distinct cod_munic
Gov table:  5570 rows, 5570 distinct cod_munic

All integrity checks passed: 5570 == 5570 == 5.570 rows


## Step 6 - Merge data tables

In [0]:
# Merge the indicator tables on cod_munic (inner join)
df_merged_indicators = df_info_indicators.join(df_gov_indicators, on="cod_munic", how="inner")

# Cast cod_munic to string as cod_ibge for the join with city metainfo
df_merged_indicators = (
    df_merged_indicators
    .withColumn("cod_ibge", F.col("cod_munic").cast("string"))
    .drop("cod_munic")
)

# Join with city metainfo on cod_ibge
df_silver_data = df_city_metainfo.join(df_merged_indicators, on="cod_ibge", how="inner")

merged_count = df_silver_data.count()

print(f"Merged table row count: {merged_count}")
print(f"Merged table column count: {len(df_silver_data.columns)}")

# An inner join silently drops rows whose key is missing on either side.
# Assert the municipality count survived both joins instead of trusting the print.
assert merged_count == 5570, f"Rows lost in the merge: expected 5.570, got {merged_count}"

Merged table row count: 5570
Merged table column count: 366


## Step 7 - Write to silver

In [0]:
# The indicator treatment in Steps 3 and 4 rebuilds the DataFrames with select(),
# which drops the lineage columns written by the bronze job. Re-attach them here so
# the silver tables still answer "where did this row come from and when".
#
# reference_year and munic_source_url come from env_variables (env notebook).

df_silver_data = df_silver_data.withColumns({
    "ano_edicao": F.lit(ano_edicao),
    "dataset_name": F.lit("IBGE_MUNIC"),
    "reference_year": F.lit(reference_year),
    "source_url": F.lit(munic_source_url),
    "source_file": F.lit(munic_file),
    "silver_processed_at": F.current_timestamp(),
})

# Place entity key first, then partition key: cod_ibge then ano_edicao
all_cols = df_silver_data.columns
other_cols = [c for c in all_cols if c not in ("ano_edicao", "cod_ibge")]
df_silver_data = df_silver_data.select("cod_ibge", "ano_edicao", *other_cols)

display(
    df_silver_data.select(
        "cod_ibge", "ano_edicao", "dataset_name", "reference_year", "silver_processed_at"
    ).limit(5)
)

ano_edicao,cod_ibge,dataset_name,reference_year,silver_processed_at
2024,1100015,IBGE_MUNIC,2024,2026-10-08T20:28:54.014Z
2024,1100023,IBGE_MUNIC,2024,2026-10-08T20:28:54.014Z
2024,1100031,IBGE_MUNIC,2024,2026-10-08T20:28:54.014Z
2024,1100049,IBGE_MUNIC,2024,2026-10-08T20:28:54.014Z
2024,1100056,IBGE_MUNIC,2024,2026-10-08T20:28:54.014Z


In [0]:
# --- Write to silver using MERGE (upsert), partitioned by ano_edicao ---
# upsert_delta_table is imported from env_functions notebook

# Clean the dictionary - place cod_variable (entity key) first, then ano_edicao (partition key)
df_clean_dict = df_raw_dict.select("cod_variable", "ano_edicao", "meaning")

# Dictionary table - merge key: ano_edicao + cod_variable
upsert_delta_table(
    df_clean_dict,
    silver_tbl_dict,
    ["ano_edicao", "cod_variable"],
    ano_edicao,
    f"Cleaned MUNIC dictionary - {ano_edicao} edition"
)

# Data table - merge key: ano_edicao + cod_ibge
upsert_delta_table(
    df_silver_data,
    silver_tbl_data,
    ["ano_edicao", "cod_ibge"],
    ano_edicao,
    f"Cleaned and merged MUNIC survey data - {ano_edicao} edition"
)

print(f"\nDict rows: {df_clean_dict.count()}")
print(f"Data rows: {df_silver_data.count()}")

MERGE done: munic_catalog.silver.munic_clean_dict - year 2024
MERGE done: munic_catalog.silver.munic_clean_data - year 2024

Dict rows: 358
Data rows: 5570


## Seing the History of the silvers_tables

In [0]:
%sql
DESCRIBE HISTORY  munic_catalog.silver.munic_clean_data

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-07T18:42:36.000Z,76129625146210,alexandre.marchioli@uel.br,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""9d6e5972-3bfc-4f6e-8c35-28fc44d60170""}, statsOnLoad -> true)",null,List(1459396063103262),2070d822-2016-4115-9f5a-f64418cd6413,1007-170921-vb1rajjs-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 483886, numDeletionVectorsRemoved -> 0, numOutputRows -> 5570, numOutputBytes -> 485982)",Cleaned and merged MUNIC survey data (Informática + Governanca),Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-07T17:56:13.000Z,71663295051113,mateus.panunci@uel.br,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""9d6e5972-3bfc-4f6e-8c35-28fc44d60170""}, statsOnLoad -> true)",null,List(3056089561309147),d9c49a3e-3d62-4644-8a0d-e75def0c66f0,1007-170921-vb1rajjs-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 591770, numDeletionVectorsRemoved -> 0, numOutputRows -> 5570, numOutputBytes -> 483886)",Cleaned and merged MUNIC survey data (Informática + Governanca),Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-07T17:12:19.000Z,71663295051113,mateus.panunci@uel.br,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""9d6e5972-3bfc-4f6e-8c35-28fc44d60170""}, statsOnLoad -> true)",null,List(3056089561309147),9447df06-42de-4914-96bf-727039c411ac,1007-170921-vb1rajjs-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5570, numOutputBytes -> 591770)",Cleaned and merged MUNIC survey data (Informática + Governanca),Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
%sql
DESCRIBE HISTORY  munic_catalog.silver.munic_clean_dict

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-07T18:42:30.000Z,76129625146210,alexandre.marchioli@uel.br,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""c3804b33-0773-4779-9cfd-847b87d5ff53""}, statsOnLoad -> true)",null,List(1459396063103262),76f574f2-5550-4a42-879a-e7700dd7649e,1007-170921-vb1rajjs-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 8359, numDeletionVectorsRemoved -> 0, numOutputRows -> 358, numOutputBytes -> 8359)",Cleaned MUNIC dictionary of variables,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-07T17:56:08.000Z,71663295051113,mateus.panunci@uel.br,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""c3804b33-0773-4779-9cfd-847b87d5ff53""}, statsOnLoad -> true)",null,List(3056089561309147),72df177d-8b3c-4e58-befc-b4303749a97f,1007-170921-vb1rajjs-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 8359, numDeletionVectorsRemoved -> 0, numOutputRows -> 358, numOutputBytes -> 8359)",Cleaned MUNIC dictionary of variables,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-07T17:12:05.000Z,71663295051113,mateus.panunci@uel.br,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""c3804b33-0773-4779-9cfd-847b87d5ff53""}, statsOnLoad -> true)",null,List(3056089561309147),8630088c-a140-4aaa-b8e8-cebc92a29e4b,1007-170921-vb1rajjs-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 358, numOutputBytes -> 8359)",Cleaned MUNIC dictionary of variables,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
